# See What Matters: Skin Lesion Robustness

Fine-tune a pretrained classifier on HAM10000 skin lesion data two ways:
1. **Baseline** -- minimal augmentation (just resize + crop)
2. **Augmented** -- targeted augmentations simulating real-world variation (skin tone, lighting, blur, noise)

Then compare both models on clean test images AND perturbed test images to measure robustness.

In [ ]:
!pip install -q torch torchvision datasets pillow matplotlib numpy scikit-learn

In [ ]:
import os
import io
import time
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Load HAM10000 data

In [ ]:
from datasets import load_dataset

print('Loading dataset (this takes a minute)...')
ds = load_dataset('ahmed-ai/skin-lesions-classification-dataset')
print(f'Train: {len(ds["train"])}, Test: {len(ds["test"])}')

# check label distribution
train_labels = [row['label'] for row in ds['train']]
print(f'Classes: {sorted(set(train_labels))}')
print(f'Distribution: {Counter(train_labels)}')

In [ ]:
# figure out class names
label_names = sorted(set(train_labels))
NUM_CLASSES = len(label_names)
label2idx = {l: i for i, l in enumerate(label_names)}
idx2label = {i: l for l, i in label2idx.items()}
print(f'{NUM_CLASSES} classes: {label_names}')

## 2. Define perturbations

In [ ]:
def shift_skin_tone(img, factor):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    lin = np.power(np.clip(arr / 255.0, 0, 1), 2.2)
    out = lin * factor
    rgb = (np.clip(np.power(out, 1/2.2), 0, 1) * 255).astype(np.uint8)
    return Image.fromarray(rgb)

def warm_lighting(img, strength=0.15):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    arr[..., 0] = np.clip(arr[..., 0] * (1 + strength), 0, 255)
    arr[..., 2] = np.clip(arr[..., 2] * (1 - strength), 0, 255)
    return Image.fromarray(arr.astype(np.uint8))

def cool_lighting(img, strength=0.15):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    arr[..., 0] = np.clip(arr[..., 0] * (1 - strength), 0, 255)
    arr[..., 2] = np.clip(arr[..., 2] * (1 + strength), 0, 255)
    return Image.fromarray(arr.astype(np.uint8))

def add_gaussian_noise(img, sigma=15.0):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    noise = np.random.normal(0, sigma, arr.shape)
    return Image.fromarray(np.clip(arr + noise, 0, 255).astype(np.uint8))

def motion_blur(img, kernel=9):
    arr = np.asarray(img.convert('RGB'), dtype=np.float32)
    k = max(3, kernel | 1)
    pad = k // 2
    padded = np.pad(arr, ((0,0),(pad,pad),(0,0)), mode='edge')
    out = np.zeros_like(arr)
    for i in range(k):
        out += padded[:, i:i+arr.shape[1], :]
    out /= k
    return Image.fromarray(out.astype(np.uint8))

def jpeg_compress(img, quality=25):
    buf = io.BytesIO()
    img.convert('RGB').save(buf, format='JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).copy()

PERTURBATIONS = {
    'darker_skin':    lambda im: shift_skin_tone(im, 0.55),
    'lighter_skin':   lambda im: shift_skin_tone(im, 1.45),
    'low_light':      lambda im: ImageEnhance.Brightness(im).enhance(0.55),
    'harsh_light':    lambda im: ImageEnhance.Brightness(ImageEnhance.Contrast(im).enhance(1.4)).enhance(1.25),
    'warm_white_bal': lambda im: warm_lighting(im, 0.20),
    'cool_white_bal': lambda im: cool_lighting(im, 0.20),
    'motion_blur':    lambda im: motion_blur(im, 11),
    'out_of_focus':   lambda im: im.filter(ImageFilter.GaussianBlur(radius=2.5)),
    'sensor_noise':   lambda im: add_gaussian_noise(im, 18.0),
    'jpeg_artifacts': lambda im: jpeg_compress(im, 20),
    'off_axis':       lambda im: im.rotate(15, resample=Image.BILINEAR, fillcolor=(0,0,0)),
}

print(f'Defined {len(PERTURBATIONS)} perturbations')

## 3. Datasets -- baseline vs augmented

In [ ]:
IMG_SIZE = 224

normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

# baseline: just resize + center crop
baseline_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    normalize,
])

# augmented: includes our domain-specific transforms
class DomainAugTransform:
    def __init__(self):
        self.to_tensor = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(IMG_SIZE),
            transforms.ToTensor(),
            normalize,
        ])

    def __call__(self, img):
        # 50% chance: apply one random domain perturbation before standard transforms
        if random.random() < 0.5:
            aug_name = random.choice(list(PERTURBATIONS.keys()))
            try:
                img = PERTURBATIONS[aug_name](img)
            except Exception:
                pass

        # standard geometric augmentations
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() < 0.3:
            angle = random.uniform(-20, 20)
            img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=(0,0,0))

        return self.to_tensor(img)

augmented_transform = DomainAugTransform()

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    normalize,
])

print('Transforms defined.')

In [ ]:
class SkinLesionDataset(Dataset):
    def __init__(self, hf_dataset, transform, label2idx):
        self.data = hf_dataset
        self.transform = transform
        self.label2idx = label2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row['image'].convert('RGB')
        label = self.label2idx[row['label']]
        img = self.transform(img)
        return img, label

train_baseline_ds = SkinLesionDataset(ds['train'], baseline_transform, label2idx)
train_augmented_ds = SkinLesionDataset(ds['train'], augmented_transform, label2idx)
test_ds = SkinLesionDataset(ds['test'], test_transform, label2idx)

BATCH_SIZE = 32
train_baseline_loader = DataLoader(train_baseline_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
train_augmented_loader = DataLoader(train_augmented_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'Train: {len(train_baseline_ds)}, Test: {len(test_ds)}')

## 4. Build model (transfer learning from MobileNetV2)

In [ ]:
def make_model(num_classes):
    net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

    # freeze the backbone
    for param in net.features.parameters():
        param.requires_grad = False

    # replace classifier head
    net.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(net.last_channel, num_classes),
    )

    return net.to(device)

print(f'Model: MobileNetV2, frozen backbone, {NUM_CLASSES}-class head')

## 5. Training loop

In [ ]:
def train_model(model, loader, num_epochs=8, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=0.3)

    history = {'loss': [], 'acc': []}

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        t0 = time.time()

        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * imgs.size(0)
            _, preds = outputs.max(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        scheduler.step()
        epoch_loss = running_loss / total
        epoch_acc = correct / total
        history['loss'].append(epoch_loss)
        history['acc'].append(epoch_acc)
        elapsed = time.time() - t0
        print(f'  Epoch {epoch+1}/{num_epochs} -- loss: {epoch_loss:.4f}, acc: {epoch_acc:.4f} ({elapsed:.1f}s)')

    return history

In [ ]:
NUM_EPOCHS = 8

print('=== Training BASELINE model (minimal augmentation) ===')
baseline_model = make_model(NUM_CLASSES)
baseline_history = train_model(baseline_model, train_baseline_loader, num_epochs=NUM_EPOCHS)

print()
print('=== Training AUGMENTED model (domain-specific augmentation) ===')
augmented_model = make_model(NUM_CLASSES)
augmented_history = train_model(augmented_model, train_augmented_loader, num_epochs=NUM_EPOCHS)

## 6. Training curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(baseline_history['loss'], label='Baseline')
ax1.plot(augmented_history['loss'], label='Augmented')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(baseline_history['acc'], label='Baseline')
ax2.plot(augmented_history['acc'], label='Augmented')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 7. Evaluate on CLEAN test set

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)
    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

print('=== Baseline on clean test set ===')
bl_labels, bl_preds, bl_probs = evaluate_model(baseline_model, test_loader)
bl_acc = accuracy_score(bl_labels, bl_preds)
print(f'Accuracy: {bl_acc:.4f}')
print(classification_report(bl_labels, bl_preds, target_names=label_names))

print('=== Augmented on clean test set ===')
aug_labels, aug_preds, aug_probs = evaluate_model(augmented_model, test_loader)
aug_acc = accuracy_score(aug_labels, aug_preds)
print(f'Accuracy: {aug_acc:.4f}')
print(classification_report(aug_labels, aug_preds, target_names=label_names))

## 8. Evaluate on PERTURBED test sets -- the main robustness comparison

In [ ]:
class PerturbedTestDataset(Dataset):
    """Apply a single perturbation to every test image before the standard transform."""
    def __init__(self, hf_dataset, perturb_fn, transform, label2idx):
        self.data = hf_dataset
        self.perturb_fn = perturb_fn
        self.transform = transform
        self.label2idx = label2idx

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data[idx]
        img = row['image'].convert('RGB')
        label = self.label2idx[row['label']]
        img = self.perturb_fn(img)
        img = self.transform(img)
        return img, label

In [ ]:
robustness_results = {}

for pert_name, pert_fn in PERTURBATIONS.items():
    print(f'Testing perturbation: {pert_name}...')
    perturbed_ds = PerturbedTestDataset(ds['test'], pert_fn, test_transform, label2idx)
    perturbed_loader = DataLoader(perturbed_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    _, bl_pert_preds, _ = evaluate_model(baseline_model, perturbed_loader)
    _, aug_pert_preds, _ = evaluate_model(augmented_model, perturbed_loader)

    bl_pert_acc = accuracy_score(bl_labels, bl_pert_preds)
    aug_pert_acc = accuracy_score(aug_labels, aug_pert_preds)

    # agreement with clean predictions (not ground truth -- measures stability)
    bl_agreement = np.mean(bl_pert_preds == bl_preds)
    aug_agreement = np.mean(aug_pert_preds == aug_preds)

    robustness_results[pert_name] = {
        'baseline_acc': bl_pert_acc,
        'augmented_acc': aug_pert_acc,
        'baseline_agreement': bl_agreement,
        'augmented_agreement': aug_agreement,
    }
    print(f'  Baseline acc: {bl_pert_acc:.4f}, Augmented acc: {aug_pert_acc:.4f}')
    print(f'  Baseline agreement: {bl_agreement:.4f}, Augmented agreement: {aug_agreement:.4f}')

print('\nDone!')

## 9. Robustness comparison charts

In [ ]:
names = list(robustness_results.keys())
bl_accs = [robustness_results[n]['baseline_acc'] for n in names]
aug_accs = [robustness_results[n]['augmented_acc'] for n in names]
bl_agree = [robustness_results[n]['baseline_agreement'] for n in names]
aug_agree = [robustness_results[n]['augmented_agreement'] for n in names]

x = np.arange(len(names))
w = 0.38

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 9))

ax1.bar(x - w/2, bl_accs, w, label='Baseline model', color='#e74c3c')
ax1.bar(x + w/2, aug_accs, w, label='Augmented model', color='#2ecc71')
ax1.axhline(bl_acc, color='#e74c3c', ls=':', alpha=0.5, label=f'Baseline clean acc ({bl_acc:.3f})')
ax1.axhline(aug_acc, color='#2ecc71', ls=':', alpha=0.5, label=f'Augmented clean acc ({aug_acc:.3f})')
ax1.set_ylabel('Accuracy on perturbed test set')
ax1.set_title('Accuracy under perturbation (higher = better)')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=35, ha='right')
ax1.legend(loc='lower right', fontsize=9)
ax1.grid(axis='y', alpha=0.3)
ax1.set_ylim(0, 1.05)

ax2.bar(x - w/2, bl_agree, w, label='Baseline model', color='#e74c3c')
ax2.bar(x + w/2, aug_agree, w, label='Augmented model', color='#2ecc71')
ax2.set_ylabel('Agreement with clean prediction')
ax2.set_title('Prediction stability (higher = more robust)')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=35, ha='right')
ax2.axhline(1.0, color='gray', ls=':', lw=0.8)
ax2.legend(loc='lower right', fontsize=9)
ax2.grid(axis='y', alpha=0.3)
ax2.set_ylim(0, 1.05)

fig.tight_layout()
plt.savefig('robustness_comparison.png', dpi=150)
plt.show()

## 10. Summary table

In [ ]:
print(f'{"Perturbation":<18} {"BL Acc":>8} {"AUG Acc":>8} {"Diff":>8} {"BL Agree":>10} {"AUG Agree":>10} {"Diff":>8}')
print('-' * 80)

for n in names:
    r = robustness_results[n]
    acc_diff = r['augmented_acc'] - r['baseline_acc']
    agr_diff = r['augmented_agreement'] - r['baseline_agreement']
    print(f'{n:<18} {r["baseline_acc"]:>8.4f} {r["augmented_acc"]:>8.4f} {acc_diff:>+8.4f} {r["baseline_agreement"]:>10.4f} {r["augmented_agreement"]:>10.4f} {agr_diff:>+8.4f}')

print('-' * 80)
avg_bl_acc = np.mean(bl_accs)
avg_aug_acc = np.mean(aug_accs)
avg_bl_agree = np.mean(bl_agree)
avg_aug_agree = np.mean(aug_agree)
print(f'{"AVERAGE":<18} {avg_bl_acc:>8.4f} {avg_aug_acc:>8.4f} {avg_aug_acc-avg_bl_acc:>+8.4f} {avg_bl_agree:>10.4f} {avg_aug_agree:>10.4f} {avg_aug_agree-avg_bl_agree:>+8.4f}')
print()
print(f'Clean test accuracy -- Baseline: {bl_acc:.4f}, Augmented: {aug_acc:.4f}')

## 11. Visualize perturbation examples

In [ ]:
sample = ds['test'][0]['image'].convert('RGB')

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

axes[0].imshow(sample)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for i, (name, fn) in enumerate(PERTURBATIONS.items()):
    ax = axes[i + 1]
    perturbed = fn(sample)
    ax.imshow(perturbed)
    ax.set_title(name, fontsize=10)
    ax.axis('off')

fig.suptitle('All perturbations applied to the same image', fontsize=14)
fig.tight_layout()
plt.savefig('perturbation_gallery.png', dpi=150)
plt.show()

## 12. Confusion matrices

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

cm_bl = confusion_matrix(bl_labels, bl_preds)
cm_aug = confusion_matrix(aug_labels, aug_preds)

ax1.imshow(cm_bl, cmap='Blues')
ax1.set_title(f'Baseline (acc={bl_acc:.3f})')
ax1.set_xlabel('Predicted')
ax1.set_ylabel('Actual')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax1.text(j, i, str(cm_bl[i][j]), ha='center', va='center', fontsize=8)

ax2.imshow(cm_aug, cmap='Greens')
ax2.set_title(f'Augmented (acc={aug_acc:.3f})')
ax2.set_xlabel('Predicted')
ax2.set_ylabel('Actual')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        ax2.text(j, i, str(cm_aug[i][j]), ha='center', va='center', fontsize=8)

fig.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()

print('All done! Download the PNG files from the left sidebar.')